# GROUP25 CENG313 — Baseline submission (LightGBM + char TF-IDF blend)
Inputs: the competition data **and** our dataset with the `src/` code. Internet: **off**.
Trains on `train.parquet` (~5-10 min on CPU) and writes `submission.csv` for the (hidden) test set.


In [ ]:
import sys, glob, shutil
from pathlib import Path

# 1) Find our `src` package among the attached Kaggle datasets
#    (the folder that contains data.py, models.py and utils.py).
pkg = None
for f in glob.glob("/kaggle/input/**/models.py", recursive=True):
    folder = Path(f).parent
    if (folder / "data.py").exists() and (folder / "utils.py").exists():
        pkg = folder
        break
if pkg is None and Path("../src").exists():      # running locally from kaggle/
    pkg = Path("../src")
assert pkg is not None, "Package not found: attach the dataset that contains our src/ files"
if Path("/kaggle/working").exists():           # on Kaggle: copy to a folder named src/ so `import src` works
    shutil.copytree(pkg, "/kaggle/working/src", dirs_exist_ok=True)
    Path("/kaggle/working/src/__init__.py").touch()
    sys.path.insert(0, "/kaggle/working")
else:
    sys.path.insert(0, str(pkg.resolve().parent))
print("package:", pkg)

# 2) Find the competition data
hits = glob.glob("/kaggle/input/**/train.parquet", recursive=True) or glob.glob("../data/train.parquet")
DATA_DIR = Path(hits[0]).parent if hits else None
print("data:", DATA_DIR)

from src.data import load_competition_data, encode_target
from src.models import FeatureModelBaseline, TfidfDiffBaseline
from src.utils import make_submission


In [ ]:
train, test = load_competition_data(DATA_DIR)
y = encode_target(train["winner"])
print(train.shape, test.shape)

lgbm = FeatureModelBaseline("lgbm").fit(train, y)
tfidf = TfidfDiffBaseline().fit(train, y)
p = 0.5 * lgbm.predict_proba(test) + 0.5 * tfidf.predict_proba(test)
sub = make_submission(test["id"], p, "submission.csv")
print(sub["winner"].value_counts())
sub.head()